# Canonical Correlation Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [75]:
# Import Python packages
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns

In [76]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [77]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [78]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [79]:
# Read in metabolomics table with top VIPs
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)

# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [80]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,Urocanic acid,Nicotine,Phenylalanine,D-TRYPTOPHAN,Glutamyltyrosine,C19H36O4 Fatty acid,C19H38O4 Fatty acid,C19H22O4 Linear diarylheptanoids,N-Oleoylethanolamine,Gln-C14:0,Sorbitane Monooleate
LAMI.RD001.D0.C1,39859.7580,0.00,970487.500,567510.060,12406.8260,611709.700,25936.0230,0.0000,16778.8380,0.0000,0.00
LAMI.RD306.D9.C2,0.0000,178911.47,1071868.200,672054.400,20433.4900,437938.200,23435.4340,6174.2900,12865.3900,0.0000,0.00
LAMI.RD304.D11.C1,36750.4000,0.00,595568.940,328151.120,9403.6520,78491.530,1793.8712,15929.3220,0.0000,5375.8525,99470.69
LAMI.RD304.D11.C2,0.0000,0.00,303845.300,152392.720,2286.2397,261996.800,8281.6990,28877.0040,15780.3090,2691.0256,157503.92
LAMI.RD304.D0.C1,3345.8184,0.00,293386.340,137201.390,0.0000,427202.060,23404.4920,36777.8320,3228.1143,3431.0034,0.00
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,0.00,579058.800,498893.780,5144.6616,122958.260,9814.8600,70822.4450,56654.0430,0.0000,0.00
LAMI.RD308.D9.C3,3076.9688,0.00,281896.700,262886.470,0.0000,65659.700,5371.9844,5548.9746,21826.1170,0.0000,0.00
LAMI.RD319.D2.C2,0.0000,0.00,255542.720,121803.890,0.0000,73231.560,1726.0502,17378.8120,19741.2460,0.0000,0.00
LAMI.RD319.D2.C3,1407.5230,0.00,59064.336,23485.404,0.0000,55986.290,0.0000,71168.3400,0.0000,0.0000,0.00


In [81]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl.T
output_path = '../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched.biom'
save_as_biom(metaB_V1V3_tbl_transposed, output_path)

In [82]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl.index)]

top_V1V3_features = [' g__Cutibacterium', ' g__uncultured', ' g__Staphylococcus',
                     ' g__Streptococcus', ' g__Corynebacterium', ' g__Lawsonella',
                     ' g__Micrococcus', ' g__Alloprevotella', ' g__Lactobacillus', ' g__Rothia']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,2362.0,0.0,94.0,292.0,175.0,91.0,37.0,18.0,1.0,50.0
LAMI.RD001.D0.C2,1808.0,2.0,158.0,374.0,352.0,120.0,52.0,27.0,165.0,43.0
LAMI.RD001.D14.C1,2234.0,2.0,83.0,240.0,492.0,253.0,30.0,30.0,60.0,12.0
LAMI.RD001.D14.C2,1761.0,0.0,137.0,446.0,456.0,151.0,16.0,100.0,26.0,31.0
LAMI.RD001.D28.C1,2367.0,11.0,118.0,293.0,365.0,217.0,22.0,38.0,17.0,14.0
...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,981.0,2759.0,21.0,0.0,2.0,5.0,0.0,0.0,0.0,0.0
LAMI.RD318.D28.C3,3218.0,538.0,5.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1900.0,1846.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,1410.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0


In [83]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
...,...,...
261,LAMI.RD317.D21.C1,Acne_L
262,LAMI.RD001.D0.C1,Healthy
263,LAMI.RD014.D14.C2,Healthy
264,LAMI.RD314.D0.C1,Acne_L


In [84]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched.T
output_path = '../Data/multi-omics/16S_V1V3-tbl_metaB-matched.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [85]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,Urocanic acid
1,Nicotine
2,Phenylalanine
3,D-TRYPTOPHAN
4,Glutamyltyrosine
5,C19H36O4 Fatty acid
6,C19H38O4 Fatty acid
7,C19H22O4 Linear diarylheptanoids
8,N-Oleoylethanolamine
9,Gln-C14:0


In [86]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [87]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,Urocanic acid,Nicotine,Phenylalanine,D-TRYPTOPHAN,Glutamyltyrosine,C19H36O4 Fatty acid,C19H38O4 Fatty acid,C19H22O4 Linear diarylheptanoids,N-Oleoylethanolamine,Gln-C14:0,Sorbitane Monooleate
LAMI.RD001.D0.C1,39859.7580,0.00,970487.500,567510.060,12406.8260,611709.700,25936.0230,0.0000,16778.8380,0.0000,0.00
LAMI.RD306.D9.C2,0.0000,178911.47,1071868.200,672054.400,20433.4900,437938.200,23435.4340,6174.2900,12865.3900,0.0000,0.00
LAMI.RD304.D11.C1,36750.4000,0.00,595568.940,328151.120,9403.6520,78491.530,1793.8712,15929.3220,0.0000,5375.8525,99470.69
LAMI.RD304.D11.C2,0.0000,0.00,303845.300,152392.720,2286.2397,261996.800,8281.6990,28877.0040,15780.3090,2691.0256,157503.92
LAMI.RD304.D0.C1,3345.8184,0.00,293386.340,137201.390,0.0000,427202.060,23404.4920,36777.8320,3228.1143,3431.0034,0.00
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,0.00,579058.800,498893.780,5144.6616,122958.260,9814.8600,70822.4450,56654.0430,0.0000,0.00
LAMI.RD308.D9.C3,3076.9688,0.00,281896.700,262886.470,0.0000,65659.700,5371.9844,5548.9746,21826.1170,0.0000,0.00
LAMI.RD319.D2.C2,0.0000,0.00,255542.720,121803.890,0.0000,73231.560,1726.0502,17378.8120,19741.2460,0.0000,0.00
LAMI.RD319.D2.C3,1407.5230,0.00,59064.336,23485.404,0.0000,55986.290,0.0000,71168.3400,0.0000,0.0000,0.00


In [88]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_tbl.index)]

top_V4_features = [' g__Staphylococcus', ' g__uncultured', ' g__Lawsonella',
                    ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia']
# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Streptococcus,g__Acinetobacter,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Rothia
LAMI.RD001.D0.C1,317.0,61.0,131.0,414.0,116.0,100.0,15.0,1.0,36.0
LAMI.RD001.D14.C1,454.0,29.0,456.0,247.0,57.0,93.0,117.0,0.0,15.0
LAMI.RD004.D0.C2,1652.0,36.0,95.0,164.0,72.0,81.0,97.0,1.0,49.0
LAMI.RD001.D0.C2,349.0,46.0,217.0,368.0,83.0,75.0,182.0,3.0,56.0
LAMI.RD011.D0.C2,2132.0,46.0,291.0,153.0,31.0,60.0,26.0,3.0,27.0
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,68.0,3599.0,54.0,1.0,1.0,4.0,0.0,1.0,0.0
LAMI.RD319.D28.C1,42.0,3525.0,96.0,7.0,5.0,0.0,0.0,0.0,2.0
LAMI.RD318.D9.C2,139.0,3441.0,23.0,11.0,35.0,12.0,2.0,10.0,1.0
LAMI.RD319.D28.C2,52.0,3627.0,54.0,1.0,0.0,2.0,1.0,0.0,0.0


In [89]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
...,...,...
260,LAMI.RD305.D0.C2,Acne_L
261,LAMI.RD317.D21.C1,Acne_L
262,LAMI.RD001.D0.C1,Healthy
264,LAMI.RD314.D0.C1,Acne_L


In [90]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl.T
output_path = '../Data/multi-omics/metabolomics-tbl_16S_V4-matched.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [91]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Perform Canonical Correlation Analysis

In [92]:
# Ensure the two datasets have the same samples (rows)
common_samples = V1V3_tbl.index.intersection(metaB_V1V3_tbl.index)
microbiome = V1V3_tbl.loc[common_samples]
metabolomics = metaB_V1V3_tbl.loc[common_samples]

# Step 1: Preprocess the data
# Apply CLR transformation to each dataset
microbiome_V1V3_clr = clr(microbiome + 1e-6)  # Add a pseudocount to avoid log of zero
metabolomics_V1_V3_clr = clr(metabolomics + 1e-6)

X_V1V3= np.array(microbiome_V1V3_clr)
Y_V1V3 = np.array(metabolomics_V1_V3_clr)

# Step 2: Perform Canonical Correlation Analysis
# Set n_components to the number of canonical dimensions to explore
n_components = min(X_V1V3.shape[1], Y_V1V3.shape[1])
cca = CCA(n_components=n_components)
cca.fit(X_V1V3, Y_V1V3)

# Get canonical variates
X_V1V3_c, Y_V1V3_c = cca.transform(X_V1V3, Y_V1V3)

# Step 3: Evaluate the results
canonical_corrs = np.corrcoef(X_V1V3_c.T, Y_V1V3_c.T).diagonal(offset=n_components)

# Print the canonical correlations
print("Canonical correlations for 16S V1-V3 and metabolomics:", canonical_corrs)

Canonical correlations for 16S V1-V3 and metabolomics: [0.8871991  0.73250832 0.72364222 0.66875978 0.64657819 0.64114535
 0.60792486 0.56139161 0.49066077 0.45112947        nan]


/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/sklearn/cross_decomposition/_pls.py:305: UserWarning: Y residual is constant at iteration 10
  warnings.warn(f"Y residual is constant at iteration {k}")
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [93]:
# Ensure the two datasets have the same samples (rows)
common_samples = V4_tbl.index.intersection(metaB_V4_tbl.index)
microbiome = V4_tbl.loc[common_samples]
metabolomics = metaB_V4_tbl.loc[common_samples]

# Step 1: Preprocess the data
# Apply CLR transformation to each dataset
microbiome_V4_clr = clr(microbiome + 1e-6)  # Add a pseudocount to avoid log of zero
metabolomics_V4_clr = clr(metabolomics + 1e-6)

X_V4= np.array(microbiome_V4_clr)
Y_V4 = np.array(metabolomics_V4_clr)

# Step 2: Perform Canonical Correlation Analysis
# Set n_components to the number of canonical dimensions to explore
n_components = min(X_V4.shape[1], Y_V4.shape[1])
cca = CCA(n_components=n_components)
cca.fit(X_V4, Y_V4)

# Get canonical variates
X_V4_c, Y_V4_c = cca.transform(X_V4, Y_V4)

# Step 3: Evaluate the results
canonical_corrs = np.corrcoef(X_V4_c.T, Y_V4_c.T).diagonal(offset=n_components)

# Print the canonical correlations
print("Canonical correlations:", canonical_corrs)

Canonical correlations: [0.88105729 0.84364678 0.80573453 0.77618683 0.76090133 0.68846603
 0.68183713 0.65029049 0.59056412 0.55867898        nan]


/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/sklearn/cross_decomposition/_pls.py:305: UserWarning: Y residual is constant at iteration 10
  warnings.warn(f"Y residual is constant at iteration {k}")
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


### Plot Figure

In [95]:
# Calculate Pearson's correlation coefficients
correlation_coefficient_1 = np.corrcoef(X_V1V3_c[:, 0], Y_V1V3_c[:, 0])[0, 1]
correlation_coefficient_2 = np.corrcoef(X_V4_c[:, 0], Y_V4_c[:, 0])[0, 1]

# Calculate best fit lines (linear regression)
v1v3_slope, v1v3_intercept = np.polyfit(X_V1V3_c[:, 0], Y_V1V3_c[:, 0], 1)
v4_slope, v4_intercept = np.polyfit(X_V4_c[:, 0], Y_V4_c[:, 0], 1)

# Generate x values for plotting regression lines
x_vals = np.linspace(-1, 2, 100)

# Calculate y values using the best fit slopes and intercepts
y_vals_v1v3 = v1v3_slope * x_vals + v1v3_intercept
y_vals_v4 = v4_slope * x_vals + v4_intercept

# Create figure
plt.figure(figsize=(6, 6))

# Scatter plot for the canonical variates with specified colors
plt.scatter(X_V1V3_c[:, 0], Y_V1V3_c[:, 0], c='#87CEEB', label='16S V1V3-MetaB matched samples',
            alpha=0.7, edgecolors='black', linewidth=0.25)
plt.plot(x_vals, y_vals_v1v3, '--', color='#1976d2', linewidth=2, label=f'Pearson r= {correlation_coefficient_1:.2f}')
plt.legend(fontsize=8)
plt.scatter(X_V4_c[:, 0], Y_V4_c[:, 0], c='#CF9FFF', label='16S V4-MetaB matched samples',
            alpha=0.7, edgecolors='black', linewidth=0.25)
plt.plot(x_vals, y_vals_v4, '--', color='#5D3FD3', linewidth=2, label=f'Pearson r= {correlation_coefficient_2:.2f}')
plt.legend(fontsize=8)
# Define tick increments
x_ticks = np.arange(-1, 2.5, 0.5)
y_ticks = np.arange(-1, 1.5, 0.5)

# Set ticks and labels
plt.xticks(x_ticks, fontsize=12)
plt.yticks(y_ticks, fontsize=12)

# Set axis limits
plt.xlim(-1, 2)
plt.ylim(-1, 1)

# Formatting the plot
plt.axhline(0, color='black', linewidth=0.5, linestyle='--')  
plt.axvline(0, color='black', linewidth=0.5, linestyle='--')  
plt.xlabel('16S Canonical Variates', fontsize=12)
plt.ylabel('MetaB Canonical Variates', fontsize=12)
plt.title('Microbiome–Metabolome Canonical Correlations', fontsize=14, pad=10)
plt.legend(loc='upper left', bbox_to_anchor=(0.34, 0.18), fontsize=8)
plt.tight_layout()

# Save the figure
plt.savefig('../Figures/multi-omics_Figures/biplot_with_correlation_V1V3_and_V4.png', dpi=600)
